# AI Referee — self-contained Colab notebook
Every project file is written by a `%%writefile` cell below, so this single notebook IS the whole repo.

**Runtime → Change runtime type → GPU (T4 / L4 / A100).** Run cells top to bottom.

Team: Sahayu R, Rishi D, Shashank R, Brian L, Abhinav K, Ishmeet S

## 1. Create folders & install dependencies

In [ ]:
!mkdir -p ai-referee/src/airef/{data,models,pose,rules} ai-referee/configs ai-referee/tests
%cd ai-referee
!pip -q install SoccerNet ultralytics av opencv-python-headless pyyaml tqdm joblib
import sys, torch
sys.path.insert(0, 'src')
print('CUDA available:', torch.cuda.is_available())

/content/ai-referee
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.9/86.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 10.5 MB/s eta 0:00:00
CUDA available: True


## 2. Project files

In [ ]:
%%writefile configs/default.yaml
# ------------------------------------------------------------------
# AI Referee — central config
# ------------------------------------------------------------------
seed: 42

data:
  root: data/soccernet/mvfouls          # unzipped: train/, valid/, test/, challenge/
  # MVFouls clips are 5 s @ 25 fps (125 frames); the foul is ~frame 75.
  start_frame: 63
  end_frame: 87
  num_frames: 16                        # frames sampled from the window
  side: 224                             # spatial resolution fed to the encoder
  max_views: 2                          # live + best replay (memory bound)

labels:
  offence_severity:                     # head 1 (4 classes)
    - "No offence"
    - "Offence + No card"
    - "Offence + Yellow card"
    - "Offence + Red card"
  action_class:                         # head 2 (8 classes)
    - "Tackling"
    - "Standing tackling"
    - "High leg"
    - "Holding"
    - "Pushing"
    - "Elbowing"
    - "Challenge"
    - "Dive"

video_model:
  backbone: mvit_v2_s                   # torchvision, Kinetics-400 pretrained
  agg: attention                        # view aggregation: attention | mean | max
  feat_dim: 400
  lr: 5.0e-5
  weight_decay: 0.001
  batch_size: 4                         # per step; use grad_accum on small GPUs
  grad_accum: 2
  epochs: 12
  freeze_backbone_epochs: 2             # linear-probe warmup
  out_dir: runs/video

pose_model:
  detector: yolov8m-pose.pt             # ultralytics weights (auto-downloaded)
  max_players: 4                        # closest players to incident kept
  classifier: gradient_boosting         # gradient_boosting | mlp
  out_dir: runs/pose

confidence:
  review_threshold: 0.5                 # < 0.5 -> flag for human review
  calibrate: true                       # temperature scaling on valid split

rules:
  pitch_length_m: 105.0
  pitch_width_m: 68.0
  offside_tolerance_m: 0.15             # measurement noise margin


Writing configs/default.yaml


In [ ]:
%%writefile src/airef/__init__.py
"""AI Referee: automated soccer violation detection from video clips."""

__version__ = "0.1.0"


Writing src/airef/__init__.py


In [ ]:
%%writefile src/airef/data/__init__.py
# (package marker)


Writing src/airef/data/__init__.py


In [ ]:
%%writefile src/airef/data/download.py
"""Download SoccerNet data (MVFouls videos + Tracking positions).

The NDA password is read from the SOCCERNET_PASSWORD environment variable —
never hardcode or commit it.

Usage:
    export SOCCERNET_PASSWORD="..."
    python -m airef.data.download --root data/soccernet --task mvfouls
    python -m airef.data.download --root data/soccernet --task tracking
"""
import argparse
import os
import sys
import zipfile
from pathlib import Path


def download_mvfouls(root: str, splits, version: str | None = None):
    from SoccerNet.Downloader import SoccerNetDownloader

    password = os.environ.get("SOCCERNET_PASSWORD")
    if not password:
        sys.exit("Set SOCCERNET_PASSWORD (from your SoccerNet NDA email).")

    dl = SoccerNetDownloader(LocalDirectory=root)
    kwargs = dict(task="mvfouls", split=list(splits), password=password)
    if version:
        kwargs["version"] = version
    dl.downloadDataTask(**kwargs)

    # Unzip each split, keeping naming conventions (train/, valid/, test/, challenge/)
    task_dir = Path(root) / "mvfouls"
    for z in sorted(task_dir.glob("*.zip")):
        target = task_dir
        print(f"Extracting {z.name} ...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(target)


def download_tracking(root: str, splits):
    from SoccerNet.Downloader import SoccerNetDownloader

    dl = SoccerNetDownloader(LocalDirectory=root)
    dl.downloadDataTask(task="tracking", split=list(splits))


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--root", default="data/soccernet")
    p.add_argument("--task", choices=["mvfouls", "tracking"], required=True)
    p.add_argument("--splits", nargs="+", default=["train", "valid", "test", "challenge"])
    p.add_argument("--version", default=None, help='e.g. "720p" for HD mvfouls clips')
    args = p.parse_args()

    if args.task == "mvfouls":
        download_mvfouls(args.root, args.splits, args.version)
    else:
        download_tracking(args.root, [s for s in args.splits if s != "valid"])


if __name__ == "__main__":
    main()


Writing src/airef/data/download.py


In [ ]:
%%writefile src/airef/data/labels.py
"""Parse SoccerNet-MVFouls annotations.json into training labels.

Each action in annotations.json has (among 10 properties):
    "Offence":       "Offence" | "No offence" | "Between" | ""
    "Severity":      "1.0" .. "5.0"  (1 = no card, 2-3 = yellow, 4-5 = red)
    "Action class":  Tackling | Standing tackling | High leg | Holding |
                     Pushing | Elbowing | Challenge | Dive
    "Bodypart", "Try to play", "Touch ball", "Multiple fouls", ...
    "Clips":         [{"Url": ..., "Camera type": ...}, ...]
"""
from __future__ import annotations

import json
from dataclasses import dataclass, field
from pathlib import Path

OFFENCE_SEVERITY_CLASSES = [
    "No offence",
    "Offence + No card",
    "Offence + Yellow card",
    "Offence + Red card",
]

ACTION_CLASSES = [
    "Tackling",
    "Standing tackling",
    "High leg",
    "Holding",
    "Pushing",
    "Elbowing",
    "Challenge",
    "Dive",
]

CARD_NAMES = {0: "No violation", 1: "No card", 2: "Yellow card", 3: "Red card"}


def offence_severity_label(offence: str, severity: str) -> int | None:
    """Map raw (Offence, Severity) annotation to a 4-class label.

    Returns None for un-annotated / ambiguous actions (dropped from training,
    following the VARS baseline).
    """
    offence = (offence or "").strip()
    severity = (severity or "").strip()
    if offence in ("No offence", "No Offence"):
        return 0
    if offence == "Offence" or offence == "Between":
        if severity in ("1.0", "2.0") or (offence == "Between"):
            return 1  # offence, no card
        if severity == "3.0":
            return 2  # yellow card
        if severity in ("4.0", "5.0"):
            return 3  # red card
    return None


def action_class_label(action_class: str) -> int | None:
    action_class = (action_class or "").strip()
    if action_class in ("", "Dont know", "Don't know"):
        return None
    try:
        return ACTION_CLASSES.index(action_class)
    except ValueError:
        return None


@dataclass
class MVFoulAction:
    action_id: str
    split: str
    clip_paths: list[str]                 # one video per camera view
    offence_severity: int | None          # 0..3
    action_class: int | None              # 0..7
    raw: dict = field(default_factory=dict, repr=False)


def load_split(data_root: str | Path, split: str) -> list[MVFoulAction]:
    """Load one split ('train' | 'valid' | 'test' | 'challenge')."""
    split_dir = Path(data_root) / split
    ann_path = split_dir / "annotations.json"
    with open(ann_path) as f:
        ann = json.load(f)

    actions: list[MVFoulAction] = []
    for action_id, a in ann["Actions"].items():
        action_dir = split_dir / f"action_{action_id}"
        clips = sorted(action_dir.glob("*.mp4")) if action_dir.exists() else []
        actions.append(
            MVFoulAction(
                action_id=action_id,
                split=split,
                clip_paths=[str(c) for c in clips],
                offence_severity=offence_severity_label(
                    a.get("Offence", ""), a.get("Severity", "")
                ),
                action_class=action_class_label(a.get("Action class", "")),
                raw=a,
            )
        )
    return actions


def trainable(actions: list[MVFoulAction]) -> list[MVFoulAction]:
    """Keep actions that have both labels and at least one clip."""
    return [
        a
        for a in actions
        if a.offence_severity is not None
        and a.action_class is not None
        and len(a.clip_paths) > 0
    ]


Writing src/airef/data/labels.py


In [ ]:
%%writefile src/airef/data/mvfouls_dataset.py
"""PyTorch Dataset for SoccerNet-MVFouls multi-view clips."""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import Dataset

from .labels import load_split, trainable

# Kinetics-400 normalization (matches MViT pretraining)
MEAN = torch.tensor([0.45, 0.45, 0.45]).view(3, 1, 1, 1)
STD = torch.tensor([0.225, 0.225, 0.225]).view(3, 1, 1, 1)


def read_clip_frames(
    path: str,
    start_frame: int,
    end_frame: int,
    num_frames: int,
    side: int,
) -> torch.Tensor:
    """Decode a clip and return a (C, T, H, W) float tensor in [0, 1]."""
    import av
    import cv2

    frames = []
    with av.open(path) as container:
        for i, frame in enumerate(container.decode(video=0)):
            if i > end_frame:
                break
            if i >= start_frame:
                frames.append(frame.to_ndarray(format="rgb24"))

    if not frames:
        raise RuntimeError(f"No frames decoded from {path}")

    idx = np.linspace(0, len(frames) - 1, num_frames).round().astype(int)
    frames = [cv2.resize(frames[i], (side, side)) for i in idx]
    video = torch.from_numpy(np.stack(frames)).permute(3, 0, 1, 2).float() / 255.0
    return (video - MEAN) / STD


class MVFoulsDataset(Dataset):
    """Yields (views, offence_severity, action_class, action_id).

    views: (V, C, T, H, W) — first clip is the live action, then replays,
    padded/truncated to `max_views`.
    """

    def __init__(
        self,
        data_root: str,
        split: str,
        start_frame: int = 63,
        end_frame: int = 87,
        num_frames: int = 16,
        side: int = 224,
        max_views: int = 2,
        train_augment: bool = False,
    ):
        self.actions = trainable(load_split(data_root, split))
        self.start_frame, self.end_frame = start_frame, end_frame
        self.num_frames, self.side, self.max_views = num_frames, side, max_views
        self.train_augment = train_augment

    def __len__(self):
        return len(self.actions)

    def _augment(self, video: torch.Tensor) -> torch.Tensor:
        if torch.rand(1).item() < 0.5:
            video = torch.flip(video, dims=[3])  # horizontal flip
        return video

    def __getitem__(self, i):
        a = self.actions[i]
        paths = a.clip_paths[: self.max_views]
        views = []
        for p in paths:
            v = read_clip_frames(
                p, self.start_frame, self.end_frame, self.num_frames, self.side
            )
            if self.train_augment:
                v = self._augment(v)
            views.append(v)
        while len(views) < self.max_views:  # pad by repeating last view
            views.append(views[-1].clone())
        return (
            torch.stack(views),
            torch.tensor(a.offence_severity, dtype=torch.long),
            torch.tensor(a.action_class, dtype=torch.long),
            a.action_id,
        )


Writing src/airef/data/mvfouls_dataset.py


In [ ]:
%%writefile src/airef/models/__init__.py
# (package marker)


Writing src/airef/models/__init__.py


In [ ]:
%%writefile src/airef/models/video_model.py
"""Pipeline A: VARS-style multi-view, multi-task video model.

Architecture (following Held et al., CVPR-W 2023):
    E: per-view video encoder (torchvision MViT-V2-S, Kinetics-400 pretrained)
    A: aggregation over camera views (attention / mean / max)
    C: two classification heads
        - offence/severity: 4 classes (no offence / no card / yellow / red)
        - action class:     8 classes (tackling, ..., dive)
"""
from __future__ import annotations

import torch
import torch.nn as nn


def build_backbone(name: str = "mvit_v2_s", pretrained: bool = True) -> tuple[nn.Module, int]:
    """Return (encoder, feature_dim). Encoder maps (B, C, T, H, W) -> (B, D)."""
    if name == "mvit_v2_s":
        from torchvision.models.video import mvit_v2_s, MViT_V2_S_Weights

        weights = MViT_V2_S_Weights.KINETICS400_V1 if pretrained else None
        m = mvit_v2_s(weights=weights)
        return m, 400  # keep the 400-d kinetics logits as features (VARS does this)
    if name == "r3d_18":
        from torchvision.models.video import r3d_18, R3D_18_Weights

        weights = R3D_18_Weights.KINETICS400_V1 if pretrained else None
        m = r3d_18(weights=weights)
        m.fc = nn.Identity()
        return m, 512
    raise ValueError(f"Unknown backbone: {name}")


class ViewAttention(nn.Module):
    """Learned attention weights over camera views."""

    def __init__(self, dim: int):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, dim // 2), nn.ReLU(), nn.Linear(dim // 2, 1))

    def forward(self, feats: torch.Tensor) -> torch.Tensor:  # (B, V, D)
        w = torch.softmax(self.score(feats), dim=1)  # (B, V, 1)
        return (w * feats).sum(dim=1)


class VARSModel(nn.Module):
    def __init__(
        self,
        backbone: str = "mvit_v2_s",
        agg: str = "attention",
        n_offence: int = 4,
        n_action: int = 8,
        pretrained: bool = True,
    ):
        super().__init__()
        self.encoder, dim = build_backbone(backbone, pretrained)
        self.agg_type = agg
        if agg == "attention":
            self.agg = ViewAttention(dim)
        self.norm = nn.LayerNorm(dim)
        self.head_offence = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(), nn.Dropout(0.3), nn.Linear(dim, n_offence)
        )
        self.head_action = nn.Sequential(
            nn.Linear(dim, dim), nn.ReLU(), nn.Dropout(0.3), nn.Linear(dim, n_action)
        )

    def encode_views(self, views: torch.Tensor) -> torch.Tensor:
        """views: (B, V, C, T, H, W) -> (B, V, D)"""
        b, v = views.shape[:2]
        flat = views.flatten(0, 1)  # (B*V, C, T, H, W)
        feats = self.encoder(flat)  # (B*V, D)
        return feats.view(b, v, -1)

    def forward(self, views: torch.Tensor):
        feats = self.encode_views(views)
        if self.agg_type == "attention":
            pooled = self.agg(feats)
        elif self.agg_type == "max":
            pooled = feats.max(dim=1).values
        else:
            pooled = feats.mean(dim=1)
        pooled = self.norm(pooled)
        return self.head_offence(pooled), self.head_action(pooled)

    def freeze_backbone(self, freeze: bool = True):
        for p in self.encoder.parameters():
            p.requires_grad = not freeze


Writing src/airef/models/video_model.py


In [ ]:
%%writefile src/airef/models/pose_model.py
"""Pipeline B: classic ML classifiers on pose features."""
from __future__ import annotations

from pathlib import Path

import joblib
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def build_classifier(kind: str = "gradient_boosting", seed: int = 42) -> Pipeline:
    if kind == "gradient_boosting":
        clf = GradientBoostingClassifier(
            n_estimators=300, max_depth=3, learning_rate=0.05,
            subsample=0.8, random_state=seed,
        )
    elif kind == "mlp":
        clf = MLPClassifier(
            hidden_layer_sizes=(256, 128), max_iter=500,
            early_stopping=True, random_state=seed,
        )
    else:
        raise ValueError(f"Unknown classifier: {kind}")
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


class PoseRefereeModel:
    """Two sklearn classifiers sharing the same pose features:
    one for offence/severity (4-class), one for action type (8-class).
    """

    def __init__(self, kind: str = "gradient_boosting", seed: int = 42):
        self.offence = build_classifier(kind, seed)
        self.action = build_classifier(kind, seed)

    def fit(self, X: np.ndarray, y_offence: np.ndarray, y_action: np.ndarray):
        self.offence.fit(X, y_offence)
        self.action.fit(X, y_action)
        return self

    def predict_proba(self, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        return self.offence.predict_proba(X), self.action.predict_proba(X)

    def save(self, path: str | Path):
        joblib.dump(self, path)

    @staticmethod
    def load(path: str | Path) -> "PoseRefereeModel":
        return joblib.load(path)


Writing src/airef/models/pose_model.py


In [ ]:
%%writefile src/airef/pose/__init__.py
# (package marker)


Writing src/airef/pose/__init__.py


In [ ]:
%%writefile src/airef/pose/extract_keypoints.py
"""Extract player pose keypoints from MVFouls clips with YOLOv8-pose.

For each action we process the FIRST clip (live view) in the foul window and
keep the `max_players` people closest to the image center at the foul frame
(the camera centers the incident). Output: one .npz per action:

    keypoints: (T, P, 17, 3)  — x, y (pixels), confidence; NaN-padded
    boxes:     (T, P, 4)

    python -m airef.pose.extract_keypoints --data data/soccernet/mvfouls --out data/keypoints
"""
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
from tqdm import tqdm

from ..data.labels import load_split, trainable

N_KP = 17  # COCO keypoints


def select_central_players(boxes: np.ndarray, img_wh: tuple[int, int], k: int) -> np.ndarray:
    """Indices of the k boxes closest to image center."""
    if len(boxes) == 0:
        return np.array([], dtype=int)
    cx, cy = img_wh[0] / 2, img_wh[1] / 2
    centers = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2], 1)
    d = np.hypot(centers[:, 0] - cx, centers[:, 1] - cy)
    return np.argsort(d)[:k]


def extract_clip(model, clip_path: str, start: int, end: int, max_players: int):
    """Run pose model on frames [start, end]; returns (T, P, 17, 3), (T, P, 4)."""
    import av

    kps_t, box_t = [], []
    with av.open(clip_path) as container:
        for i, frame in enumerate(container.decode(video=0)):
            if i > end:
                break
            if i < start:
                continue
            img = frame.to_ndarray(format="rgb24")
            res = model(img, verbose=False)[0]
            kps = np.full((max_players, N_KP, 3), np.nan, dtype=np.float32)
            boxes = np.full((max_players, 4), np.nan, dtype=np.float32)
            if res.keypoints is not None and len(res.boxes) > 0:
                raw_boxes = res.boxes.xyxy.cpu().numpy()
                raw_kps = res.keypoints.data.cpu().numpy()  # (N, 17, 3)
                keep = select_central_players(
                    raw_boxes, (img.shape[1], img.shape[0]), max_players
                )
                for j, idx in enumerate(keep):
                    kps[j] = raw_kps[idx]
                    boxes[j] = raw_boxes[idx]
            kps_t.append(kps)
            box_t.append(boxes)
    return np.stack(kps_t), np.stack(box_t)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True, help="mvfouls root (train/valid/test dirs)")
    p.add_argument("--out", required=True)
    p.add_argument("--splits", nargs="+", default=["train", "valid", "test"])
    p.add_argument("--weights", default="yolov8m-pose.pt")
    p.add_argument("--start-frame", type=int, default=63)
    p.add_argument("--end-frame", type=int, default=87)
    p.add_argument("--max-players", type=int, default=4)
    args = p.parse_args()

    from ultralytics import YOLO

    model = YOLO(args.weights)

    for split in args.splits:
        out_dir = Path(args.out) / split
        out_dir.mkdir(parents=True, exist_ok=True)
        actions = trainable(load_split(args.data, split))
        for a in tqdm(actions, desc=split):
            out_path = out_dir / f"action_{a.action_id}.npz"
            if out_path.exists():
                continue
            try:
                kps, boxes = extract_clip(
                    model, a.clip_paths[0], args.start_frame, args.end_frame, args.max_players
                )
                np.savez_compressed(
                    out_path, keypoints=kps, boxes=boxes,
                    offence_severity=a.offence_severity, action_class=a.action_class,
                )
            except Exception as e:  # noqa: BLE001 — skip corrupt clips, keep going
                print(f"skip action_{a.action_id}: {e}")


if __name__ == "__main__":
    main()


Writing src/airef/pose/extract_keypoints.py


In [ ]:
%%writefile src/airef/pose/features.py
"""Turn pose keypoint sequences into fixed-length feature vectors.

Input:  keypoints (T, P, 17, 3) — x, y, conf; NaN where no detection.
Output: 1-D feature vector capturing motion, contact, and limb geometry —
the signals a referee actually looks at:

- per-player speed / acceleration statistics (lunging, charging)
- minimum & mean inter-player distance over time (contact proxy)
- closing speed between the two central players (intensity of challenge)
- ankle/knee height relative to hip (high-leg / studs-up indicator)
- elbow height relative to shoulder (elbowing indicator)
- torso lean angle (diving / off-balance indicator)

COCO keypoint indices used:
    5/6 shoulders, 7/8 elbows, 11/12 hips, 13/14 knees, 15/16 ankles
"""
from __future__ import annotations

import numpy as np

L_SHO, R_SHO = 5, 6
L_ELB, R_ELB = 7, 8
L_HIP, R_HIP = 11, 12
L_KNE, R_KNE = 13, 14
L_ANK, R_ANK = 15, 16


def _stats(x: np.ndarray) -> list[float]:
    """[mean, std, min, max] ignoring NaNs; zeros if empty."""
    x = x[np.isfinite(x)]
    if x.size == 0:
        return [0.0, 0.0, 0.0, 0.0]
    return [float(np.mean(x)), float(np.std(x)), float(np.min(x)), float(np.max(x))]


def _center(kps: np.ndarray) -> np.ndarray:
    """Hip midpoint per frame: (T, 17, 3) -> (T, 2)."""
    return np.nanmean(kps[:, [L_HIP, R_HIP], :2], axis=1)


def _body_scale(kps: np.ndarray) -> float:
    """Median shoulder-to-hip distance — normalizes for camera zoom."""
    sho = np.nanmean(kps[:, [L_SHO, R_SHO], :2], axis=1)
    hip = np.nanmean(kps[:, [L_HIP, R_HIP], :2], axis=1)
    d = np.linalg.norm(sho - hip, axis=1)
    d = d[np.isfinite(d)]
    return float(np.median(d)) if d.size else 1.0


def player_features(kps: np.ndarray, scale: float) -> list[float]:
    """Features for one player: (T, 17, 3) -> list of floats."""
    feats: list[float] = []
    c = _center(kps) / scale

    vel = np.linalg.norm(np.diff(c, axis=0), axis=1)          # speed
    acc = np.diff(vel)                                        # acceleration
    feats += _stats(vel) + _stats(acc)

    hip_y = np.nanmean(kps[:, [L_HIP, R_HIP], 1], axis=1)
    for idx_pair in ([L_ANK, R_ANK], [L_KNE, R_KNE], [L_ELB, R_ELB]):
        part_y = np.nanmin(kps[:, idx_pair, 1], axis=1)       # image y grows downward
        rel = (hip_y - part_y) / scale                        # >0 == raised above hip
        feats += _stats(rel)

    sho_y = np.nanmean(kps[:, [L_SHO, R_SHO], 1], axis=1)
    elb_y = np.nanmin(kps[:, [L_ELB, R_ELB], 1], axis=1)
    feats += _stats((sho_y - elb_y) / scale)                  # elbow above shoulder

    sho_c = np.nanmean(kps[:, [L_SHO, R_SHO], :2], axis=1)
    hip_c = np.nanmean(kps[:, [L_HIP, R_HIP], :2], axis=1)
    torso = sho_c - hip_c
    lean = np.abs(np.arctan2(torso[:, 0], -torso[:, 1]))      # 0 = upright
    feats += _stats(lean)

    return feats


def pairwise_features(kps_a: np.ndarray, kps_b: np.ndarray, scale: float) -> list[float]:
    """Interaction features between the two most central players."""
    ca, cb = _center(kps_a) / scale, _center(kps_b) / scale
    dist = np.linalg.norm(ca - cb, axis=1)
    closing = -np.diff(dist)                                  # >0 == approaching
    feats = _stats(dist) + _stats(closing)
    # ankle of A vs body center of B — kicking-range proxy
    ank_a = np.nanmean(kps_a[:, [L_ANK, R_ANK], :2], axis=1) / scale
    feats += _stats(np.linalg.norm(ank_a - cb, axis=1))
    return feats


def action_features(keypoints: np.ndarray, max_players: int = 2) -> np.ndarray:
    """(T, P, 17, 3) -> fixed-length vector. Uses the two most central players."""
    P = min(max_players, keypoints.shape[1])
    scale = max(_body_scale(keypoints[:, 0]), 1e-6)
    feats: list[float] = []
    for p in range(P):
        feats += player_features(keypoints[:, p], scale)
    while P < max_players:  # pad if fewer players detected
        feats += [0.0] * len(player_features(keypoints[:, 0], scale))
        P += 1
    if keypoints.shape[1] >= 2:
        feats += pairwise_features(keypoints[:, 0], keypoints[:, 1], scale)
    else:
        feats += [0.0] * 12
    out = np.asarray(feats, dtype=np.float32)
    out[~np.isfinite(out)] = 0.0
    return out


FEATURE_NAMES_NOTE = (
    "Per player: speed[4], accel[4], ankle-above-hip[4], knee-above-hip[4], "
    "elbow-above-hip[4], elbow-above-shoulder[4], torso-lean[4]; "
    "pair: distance[4], closing-speed[4], kick-range[4]. "
    "Each [4] = mean/std/min/max."
)


Writing src/airef/pose/features.py


In [ ]:
%%writefile src/airef/rules/__init__.py
# (package marker)


Writing src/airef/rules/__init__.py


In [ ]:
%%writefile src/airef/rules/pitch.py
"""Pitch geometry in metric coordinates.

Convention: origin at pitch center, x along the length (attacking direction =
+x for the team in possession), y along the width. Positions from SoccerNet
Tracking must be homography-projected into this frame first (the tracking
dev kit provides calibration; see sn-tracking / sn-calibration repos).
"""
from dataclasses import dataclass


@dataclass(frozen=True)
class Pitch:
    length_m: float = 105.0
    width_m: float = 68.0

    @property
    def x_max(self) -> float:
        return self.length_m / 2

    @property
    def y_max(self) -> float:
        return self.width_m / 2

    def inside(self, x: float, y: float, margin: float = 0.0) -> bool:
        return abs(x) <= self.x_max + margin and abs(y) <= self.y_max + margin


Writing src/airef/rules/pitch.py


In [ ]:
%%writefile src/airef/rules/offside.py
"""Geometric offside detection from tracked player/ball positions.

Offside is deterministic given positions, so we compute it directly instead
of training a classifier — and the explanation falls out of the geometry.

Law 11: a player is in an offside POSITION if, at the moment the ball is
played by a teammate, any part of their head/body/feet is nearer to the
opponents' goal line than both the ball and the second-last opponent — unless
they are in their own half. They are only PENALIZED if they become involved
in active play.

Inputs use the metric pitch frame from `pitch.py`, with the attacking team
moving toward +x. Confidence is derived from the margin vs. tracking noise.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np

from .pitch import Pitch


@dataclass
class OffsideResult:
    offside: bool
    margin_m: float               # + = beyond the offside line
    confidence: float             # 0..1, from margin vs. tolerance
    needs_human_review: bool
    offside_line_x: float
    explanation: str
    per_attacker: list[dict] = field(default_factory=list)


def _sigmoid(z: float) -> float:
    return float(1.0 / (1.0 + np.exp(-z)))


def check_offside(
    attackers_xy: np.ndarray,        # (A, 2) attacking players (excl. ball carrier)
    defenders_xy: np.ndarray,        # (D, 2) defending players (incl. goalkeeper)
    ball_xy: np.ndarray,             # (2,) at the moment of the pass
    pitch: Pitch = Pitch(),
    tolerance_m: float = 0.15,
    involved_mask: np.ndarray | None = None,  # (A,) bool: active in play
) -> OffsideResult:
    attackers_xy = np.atleast_2d(np.asarray(attackers_xy, dtype=float))
    defenders_xy = np.atleast_2d(np.asarray(defenders_xy, dtype=float))
    ball_xy = np.asarray(ball_xy, dtype=float)

    if len(defenders_xy) < 2:
        return OffsideResult(
            False, 0.0, 0.0, True, float("nan"),
            "Fewer than two defenders tracked — cannot establish the offside "
            "line. Flagged for human review.",
        )

    # Second-last defender (defenders retreat toward +x goal they defend)
    def_x_sorted = np.sort(defenders_xy[:, 0])[::-1]
    second_last_def_x = def_x_sorted[1]
    # Offside line: max(second-last defender, ball); never in own half
    line_x = max(second_last_def_x, ball_xy[0], 0.0)

    if involved_mask is None:
        involved_mask = np.ones(len(attackers_xy), dtype=bool)

    per_attacker = []
    worst_margin, offside_any = -np.inf, False
    for i, (x, y) in enumerate(attackers_xy):
        margin = x - line_x
        is_off = bool(margin > tolerance_m and involved_mask[i] and x > 0)
        per_attacker.append(
            {"attacker": i, "x": float(x), "y": float(y),
             "margin_m": float(margin), "offside": is_off}
        )
        if involved_mask[i] and margin > worst_margin:
            worst_margin = margin
        offside_any |= is_off

    # Confidence: how far the decisive margin sits outside the noise band.
    conf = _sigmoid(abs(worst_margin) / max(tolerance_m, 1e-6) - 1.0)
    needs_review = conf < 0.5

    if offside_any:
        worst = max((p for p in per_attacker if p["offside"]), key=lambda p: p["margin_m"])
        explanation = (
            f"Offside: attacker {worst['attacker']} was {worst['margin_m']:.2f} m "
            f"beyond the offside line (x = {line_x:.2f} m, set by the "
            f"{'second-last defender' if line_x == second_last_def_x else 'ball'}) "
            f"when the ball was played."
        )
    else:
        explanation = (
            f"No offside: every involved attacker was level with or behind the "
            f"offside line (x = {line_x:.2f} m); closest margin "
            f"{worst_margin:+.2f} m."
        )
    if needs_review:
        explanation += (
            f" Margin is within tracking tolerance (±{tolerance_m:.2f} m) — "
            "flagged for human review."
        )

    return OffsideResult(
        offside=offside_any,
        margin_m=float(worst_margin),
        confidence=round(conf, 3),
        needs_human_review=needs_review,
        offside_line_x=float(line_x),
        explanation=explanation,
        per_attacker=per_attacker,
    )


Writing src/airef/rules/offside.py


In [ ]:
%%writefile src/airef/rules/out_of_bounds.py
"""Out-of-bounds detection from ball tracking.

The WHOLE ball must cross the WHOLE line (Law 9), so we test against the
pitch boundary plus the ball radius. Confidence comes from the crossing
margin vs. tracking noise.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .pitch import Pitch

BALL_RADIUS_M = 0.11


@dataclass
class OutOfBoundsResult:
    out: bool
    boundary: str | None          # "touchline" | "goal line" | None
    restart: str | None           # throw-in / goal kick or corner
    margin_m: float
    confidence: float
    needs_human_review: bool
    frame: int | None
    explanation: str


def _sigmoid(z: float) -> float:
    return float(1.0 / (1.0 + np.exp(-z)))


def check_out_of_bounds(
    ball_xy_seq: np.ndarray,          # (T, 2) ball trajectory, metric frame
    pitch: Pitch = Pitch(),
    tolerance_m: float = 0.10,
) -> OutOfBoundsResult:
    seq = np.atleast_2d(np.asarray(ball_xy_seq, dtype=float))
    x_lim = pitch.x_max + BALL_RADIUS_M
    y_lim = pitch.y_max + BALL_RADIUS_M

    over_x = np.abs(seq[:, 0]) - x_lim     # + = fully past the goal line
    over_y = np.abs(seq[:, 1]) - y_lim     # + = fully past the touchline
    margin = np.maximum(over_x, over_y)
    worst_t = int(np.argmax(margin))
    worst = float(margin[worst_t])
    out = worst > 0

    boundary = restart = None
    if out:
        if over_x[worst_t] >= over_y[worst_t]:
            boundary, restart = "goal line", "goal kick or corner kick"
        else:
            boundary, restart = "touchline", "throw-in"

    conf = _sigmoid(abs(worst) / max(tolerance_m, 1e-6) - 1.0)
    needs_review = conf < 0.5

    if out:
        explanation = (
            f"Ball out of play: the whole ball crossed the {boundary} by "
            f"{worst:.2f} m at frame {worst_t}. Restart: {restart}."
        )
    else:
        explanation = (
            f"Ball in play: it never fully crossed a boundary "
            f"(closest {-worst:.2f} m inside, frame {worst_t})."
        )
    if needs_review:
        explanation += (
            f" Margin is within tracking tolerance (±{tolerance_m:.2f} m) — "
            "flagged for human review."
        )

    return OutOfBoundsResult(
        out=out, boundary=boundary, restart=restart,
        margin_m=round(worst, 3), confidence=round(conf, 3),
        needs_human_review=needs_review, frame=worst_t,
        explanation=explanation,
    )


Writing src/airef/rules/out_of_bounds.py


In [ ]:
%%writefile src/airef/confidence.py
"""Confidence calibration (temperature scaling) + human-review flagging.

Raw softmax probabilities are over-confident; temperature scaling (Guo et
al., 2017) fits a single scalar T on the validation split so that the
reported confidence matches empirical accuracy. Flag rule: calibrated
confidence < threshold (default 0.5) -> needs_human_review.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn.functional as F

REVIEW_THRESHOLD = 0.5


def fit_temperature(logits: torch.Tensor, labels: torch.Tensor, max_iter: int = 200) -> float:
    """Fit T minimizing NLL of softmax(logits / T) on held-out data."""
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=max_iter)

    def closure():
        opt.zero_grad()
        loss = F.cross_entropy(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().item())


def calibrated_probs(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    return F.softmax(logits / temperature, dim=-1)


def decide(probs: np.ndarray, threshold: float = REVIEW_THRESHOLD) -> dict:
    """Turn a probability vector into {pred, confidence, needs_human_review}."""
    probs = np.asarray(probs, dtype=float).ravel()
    pred = int(probs.argmax())
    conf = float(probs[pred])
    return {
        "pred": pred,
        "confidence": round(conf, 3),
        "needs_human_review": conf < threshold,
    }


def expected_calibration_error(
    probs: np.ndarray, labels: np.ndarray, n_bins: int = 10
) -> float:
    """ECE — reported in evaluate.py so you can show calibration worked."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (conf > lo) & (conf <= hi)
        if mask.sum():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)


Writing src/airef/confidence.py


In [ ]:
%%writefile src/airef/explain.py
"""Plain-language explanations for each call.

Foul explanations are template-based, driven by the model's two predictions
(action type + severity) and — when pose keypoints are available — motion
facts extracted from the pose features (contact distance, raised boot,
elbow height, closing speed). Offside/out-of-bounds explanations are
generated directly by the geometric rule modules.
"""
from __future__ import annotations

import numpy as np

from .data.labels import ACTION_CLASSES, OFFENCE_SEVERITY_CLASSES

_ACTION_PHRASES = {
    "Tackling": "a sliding challenge on the opponent",
    "Standing tackling": "a standing challenge for the ball",
    "High leg": "a raised boot near the opponent",
    "Holding": "holding or pulling the opponent",
    "Pushing": "a push on the opponent",
    "Elbowing": "leading with the arm or elbow",
    "Challenge": "a physical challenge for the ball",
    "Dive": "a fall without sufficient contact (simulation)",
}

_SEVERITY_PHRASES = {
    0: "the contact was fair or negligible, so no offence is called",
    1: "it is a careless offence: a free kick but no card",
    2: "it is a reckless offence, meriting a yellow card",
    3: "it involves excessive force or endangers an opponent, meriting a red card",
}


def _pose_facts(keypoints: np.ndarray | None) -> list[str]:
    """Human-readable motion facts from (T, P, 17, 3) keypoints."""
    if keypoints is None or keypoints.shape[1] < 2:
        return []
    from .pose.features import _body_scale, _center, L_ANK, R_ANK, L_HIP, R_HIP

    facts = []
    scale = max(_body_scale(keypoints[:, 0]), 1e-6)
    c0, c1 = _center(keypoints[:, 0]) / scale, _center(keypoints[:, 1]) / scale
    dist = np.linalg.norm(c0 - c1, axis=1)
    dist = dist[np.isfinite(dist)]
    if dist.size:
        if np.min(dist) < 1.0:
            facts.append("the players came into close contact")
        closing = -np.diff(dist)
        if closing.size and np.max(closing) > 0.15:
            facts.append("one player closed on the other at speed")
    hip_y = np.nanmean(keypoints[:, 0, [L_HIP, R_HIP], 1], axis=1)
    ank_y = np.nanmin(keypoints[:, 0, [L_ANK, R_ANK], 1], axis=1)
    rel = (hip_y - ank_y) / scale
    rel = rel[np.isfinite(rel)]
    if rel.size and np.max(rel) > 0.3:
        facts.append("a boot was raised above hip height")
    return facts


def explain_foul(
    offence_pred: int,
    action_pred: int,
    offence_conf: float,
    action_conf: float,
    keypoints: np.ndarray | None = None,
) -> str:
    action_name = ACTION_CLASSES[action_pred]
    parts = [
        f"The incident is classified as {_ACTION_PHRASES[action_name]} "
        f"({action_name.lower()}, confidence {action_conf:.0%})."
    ]
    facts = _pose_facts(keypoints)
    if facts:
        parts.append("From player positioning and movement, " + "; ".join(facts) + ".")
    parts.append(
        f"Based on the intensity and nature of the contact, "
        f"{_SEVERITY_PHRASES[offence_pred]} "
        f"(decision: {OFFENCE_SEVERITY_CLASSES[offence_pred]}, "
        f"confidence {offence_conf:.0%})."
    )
    if min(offence_conf, action_conf) < 0.5:
        parts.append(
            "Confidence is below 0.5, so this call is flagged for human review."
        )
    return " ".join(parts)


Writing src/airef/explain.py


In [ ]:
%%writefile src/airef/train_video.py
"""Train Pipeline A (VARS-style video model) on SoccerNet-MVFouls.

    python -m airef.train_video --config configs/default.yaml --data data/soccernet/mvfouls
"""
from __future__ import annotations

import argparse
import json
from collections import Counter
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import yaml
from torch.utils.data import DataLoader
from tqdm import tqdm

from .confidence import fit_temperature
from .data.mvfouls_dataset import MVFoulsDataset
from .models.video_model import VARSModel


def class_weights(labels: list[int], n: int) -> torch.Tensor:
    """Inverse-frequency weights (MVFouls is heavily imbalanced)."""
    counts = Counter(labels)
    w = torch.tensor([1.0 / max(counts.get(c, 1), 1) for c in range(n)], dtype=torch.float)
    return w * n / w.sum()


@torch.no_grad()
def evaluate_epoch(model, loader, device):
    model.eval()
    correct_o = correct_a = total = 0
    logits_o_all, logits_a_all, y_o_all, y_a_all = [], [], [], []
    for views, y_off, y_act, _ in loader:
        views = views.to(device)
        lo, la = model(views)
        logits_o_all.append(lo.cpu())
        logits_a_all.append(la.cpu())
        y_o_all.append(y_off)
        y_a_all.append(y_act)
        correct_o += (lo.argmax(1).cpu() == y_off).sum().item()
        correct_a += (la.argmax(1).cpu() == y_act).sum().item()
        total += len(y_off)
    return {
        "acc_offence": correct_o / total,
        "acc_action": correct_a / total,
        "logits_offence": torch.cat(logits_o_all),
        "logits_action": torch.cat(logits_a_all),
        "y_offence": torch.cat(y_o_all),
        "y_action": torch.cat(y_a_all),
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--config", default="configs/default.yaml")
    p.add_argument("--data", default=None, help="overrides data.root")
    p.add_argument("--epochs", type=int, default=None)
    args = p.parse_args()

    cfg = yaml.safe_load(open(args.config))
    d, m = cfg["data"], cfg["video_model"]
    data_root = args.data or d["root"]
    epochs = args.epochs or m["epochs"]
    torch.manual_seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"
    out_dir = Path(m["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    ds_kwargs = dict(
        start_frame=d["start_frame"], end_frame=d["end_frame"],
        num_frames=d["num_frames"], side=d["side"], max_views=d["max_views"],
    )
    train_ds = MVFoulsDataset(data_root, "train", train_augment=True, **ds_kwargs)
    valid_ds = MVFoulsDataset(data_root, "valid", **ds_kwargs)
    print(f"train: {len(train_ds)} actions | valid: {len(valid_ds)} actions | device: {device}")

    train_dl = DataLoader(train_ds, batch_size=m["batch_size"], shuffle=True,
                          num_workers=2, pin_memory=True)
    valid_dl = DataLoader(valid_ds, batch_size=m["batch_size"], num_workers=2)

    model = VARSModel(backbone=m["backbone"], agg=m["agg"]).to(device)

    w_off = class_weights([a.offence_severity for a in train_ds.actions], 4).to(device)
    w_act = class_weights([a.action_class for a in train_ds.actions], 8).to(device)
    crit_off = nn.CrossEntropyLoss(weight=w_off)
    crit_act = nn.CrossEntropyLoss(weight=w_act)

    opt = torch.optim.AdamW(model.parameters(), lr=float(m["lr"]),
                            weight_decay=float(m["weight_decay"]))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")

    best = 0.0
    for epoch in range(epochs):
        model.freeze_backbone(epoch < m["freeze_backbone_epochs"])
        model.train()
        running = 0.0
        opt.zero_grad()
        for step, (views, y_off, y_act, _) in enumerate(tqdm(train_dl, desc=f"epoch {epoch}")):
            views, y_off, y_act = views.to(device), y_off.to(device), y_act.to(device)
            with torch.autocast(device_type=device, enabled=device == "cuda"):
                lo, la = model(views)
                loss = (crit_off(lo, y_off) + crit_act(la, y_act)) / m["grad_accum"]
            scaler.scale(loss).backward()
            if (step + 1) % m["grad_accum"] == 0:
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            running += loss.item() * m["grad_accum"]
        sched.step()

        val = evaluate_epoch(model, valid_dl, device)
        score = (val["acc_offence"] + val["acc_action"]) / 2
        print(f"epoch {epoch}: loss={running / len(train_dl):.4f} "
              f"val_acc_offence={val['acc_offence']:.3f} val_acc_action={val['acc_action']:.3f}")

        if score > best:
            best = score
            torch.save(model.state_dict(), out_dir / "best.pth")

            # Calibrate confidences on the validation split
            t_off = fit_temperature(val["logits_offence"], val["y_offence"])
            t_act = fit_temperature(val["logits_action"], val["y_action"])
            with open(out_dir / "calibration.json", "w") as f:
                json.dump({"temperature_offence": t_off, "temperature_action": t_act}, f)
            print(f"  saved best (score={best:.3f}, T_off={t_off:.2f}, T_act={t_act:.2f})")

    print(f"done. best checkpoint: {out_dir / 'best.pth'}")


if __name__ == "__main__":
    main()


Writing src/airef/train_video.py


In [ ]:
%%writefile src/airef/train_pose.py
"""Train Pipeline B (pose features + classic ML).

    python -m airef.train_pose --config configs/default.yaml --keypoints data/keypoints
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import yaml
from sklearn.metrics import balanced_accuracy_score, classification_report

from .data.labels import ACTION_CLASSES, OFFENCE_SEVERITY_CLASSES
from .models.pose_model import PoseRefereeModel
from .pose.features import action_features


def load_features(keypoints_dir: str | Path, split: str):
    X, y_off, y_act, ids = [], [], [], []
    for f in sorted(Path(keypoints_dir, split).glob("action_*.npz")):
        z = np.load(f)
        X.append(action_features(z["keypoints"]))
        y_off.append(int(z["offence_severity"]))
        y_act.append(int(z["action_class"]))
        ids.append(f.stem)
    return np.stack(X), np.array(y_off), np.array(y_act), ids


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--config", default="configs/default.yaml")
    p.add_argument("--keypoints", required=True)
    args = p.parse_args()

    cfg = yaml.safe_load(open(args.config))
    out_dir = Path(cfg["pose_model"]["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    X_tr, yo_tr, ya_tr, _ = load_features(args.keypoints, "train")
    X_va, yo_va, ya_va, _ = load_features(args.keypoints, "valid")
    print(f"train: {X_tr.shape} | valid: {X_va.shape}")

    model = PoseRefereeModel(cfg["pose_model"]["classifier"], cfg["seed"])
    model.fit(X_tr, yo_tr, ya_tr)

    po, pa = model.predict_proba(X_va)
    metrics = {
        "balanced_acc_offence": balanced_accuracy_score(yo_va, po.argmax(1)),
        "balanced_acc_action": balanced_accuracy_score(ya_va, pa.argmax(1)),
    }
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        yo_va, po.argmax(1),
        labels=range(4), target_names=OFFENCE_SEVERITY_CLASSES, zero_division=0,
    ))
    print(classification_report(
        ya_va, pa.argmax(1),
        labels=range(8), target_names=ACTION_CLASSES, zero_division=0,
    ))

    model.save(out_dir / "pose_model.joblib")
    with open(out_dir / "valid_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"saved {out_dir / 'pose_model.joblib'}")


if __name__ == "__main__":
    main()


Writing src/airef/train_pose.py


In [ ]:
%%writefile src/airef/evaluate.py
"""Evaluate & compare Pipeline A (video) vs Pipeline B (pose) on the test split.

    python -m airef.evaluate --config configs/default.yaml \
        --data data/soccernet/mvfouls --keypoints data/keypoints

Reports per-task balanced accuracy, macro-F1, ECE (calibration), and the
review-flag rate at the 0.5 threshold.
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np
import torch
import yaml
from sklearn.metrics import balanced_accuracy_score, f1_score
from torch.utils.data import DataLoader

from .confidence import calibrated_probs, expected_calibration_error


def metrics_block(probs: np.ndarray, y: np.ndarray, threshold: float) -> dict:
    pred = probs.argmax(1)
    conf = probs.max(1)
    return {
        "balanced_accuracy": round(balanced_accuracy_score(y, pred), 4),
        "macro_f1": round(f1_score(y, pred, average="macro"), 4),
        "ece": round(expected_calibration_error(probs, y), 4),
        "review_flag_rate": round(float((conf < threshold).mean()), 4),
        "accuracy_on_confident": round(
            float((pred[conf >= threshold] == y[conf >= threshold]).mean())
            if (conf >= threshold).any() else float("nan"), 4),
    }


def eval_video(cfg, data_root, split, threshold):
    from .data.mvfouls_dataset import MVFoulsDataset
    from .models.video_model import VARSModel

    d, m = cfg["data"], cfg["video_model"]
    out_dir = Path(m["out_dir"])
    device = "cuda" if torch.cuda.is_available() else "cpu"

    ds = MVFoulsDataset(data_root, split,
                        start_frame=d["start_frame"], end_frame=d["end_frame"],
                        num_frames=d["num_frames"], side=d["side"],
                        max_views=d["max_views"])
    dl = DataLoader(ds, batch_size=m["batch_size"], num_workers=2)

    model = VARSModel(backbone=m["backbone"], agg=m["agg"])
    model.load_state_dict(torch.load(out_dir / "best.pth", map_location=device))
    model.to(device).eval()

    cal_path = out_dir / "calibration.json"
    t_off = t_act = 1.0
    if cal_path.exists():
        cal = json.load(open(cal_path))
        t_off, t_act = cal["temperature_offence"], cal["temperature_action"]

    P_off, P_act, Y_off, Y_act = [], [], [], []
    with torch.no_grad():
        for views, y_off, y_act, _ in dl:
            lo, la = model(views.to(device))
            P_off.append(calibrated_probs(lo, t_off).cpu().numpy())
            P_act.append(calibrated_probs(la, t_act).cpu().numpy())
            Y_off.append(y_off.numpy())
            Y_act.append(y_act.numpy())
    P_off, P_act = np.concatenate(P_off), np.concatenate(P_act)
    Y_off, Y_act = np.concatenate(Y_off), np.concatenate(Y_act)
    return {
        "offence_severity": metrics_block(P_off, Y_off, threshold),
        "action_class": metrics_block(P_act, Y_act, threshold),
    }


def eval_pose(cfg, keypoints_dir, split, threshold):
    from .models.pose_model import PoseRefereeModel
    from .train_pose import load_features

    model = PoseRefereeModel.load(Path(cfg["pose_model"]["out_dir"]) / "pose_model.joblib")
    X, y_off, y_act, _ = load_features(keypoints_dir, split)
    p_off, p_act = model.predict_proba(X)
    return {
        "offence_severity": metrics_block(p_off, y_off, threshold),
        "action_class": metrics_block(p_act, y_act, threshold),
    }


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--config", default="configs/default.yaml")
    p.add_argument("--data", default=None, help="mvfouls root (pipeline A)")
    p.add_argument("--keypoints", default=None, help="keypoints root (pipeline B)")
    p.add_argument("--split", default="test")
    args = p.parse_args()

    cfg = yaml.safe_load(open(args.config))
    threshold = cfg["confidence"]["review_threshold"]

    report: dict = {"split": args.split, "review_threshold": threshold}
    if args.data:
        report["pipeline_A_video"] = eval_video(cfg, args.data, args.split, threshold)
    if args.keypoints:
        report["pipeline_B_pose"] = eval_pose(cfg, args.keypoints, args.split, threshold)

    print(json.dumps(report, indent=2))
    out = Path("runs") / f"comparison_{args.split}.json"
    out.parent.mkdir(exist_ok=True)
    with open(out, "w") as f:
        json.dump(report, f, indent=2)
    print(f"saved {out}")


if __name__ == "__main__":
    main()


Writing src/airef/evaluate.py


In [ ]:
%%writefile src/airef/inference.py
"""Single-clip inference: video in -> refereeing decision JSON out.

    python -m airef.inference --clip incident.mp4 \
        --checkpoint runs/video/best.pth --calibration runs/video/calibration.json

Optionally pass --pose-model runs/pose/pose_model.joblib to also report
Pipeline B's opinion, and tracking positions (npz with attackers/defenders/
ball) for offside / out-of-bounds checks.
"""
from __future__ import annotations

import argparse
import json

import numpy as np
import torch
import yaml

from .confidence import calibrated_probs, decide
from .data.labels import ACTION_CLASSES, CARD_NAMES, OFFENCE_SEVERITY_CLASSES
from .data.mvfouls_dataset import read_clip_frames
from .explain import explain_foul
from .models.video_model import VARSModel


def predict_clip(
    clip_paths: list[str],
    checkpoint: str,
    calibration: str | None = None,
    config: str = "configs/default.yaml",
    device: str | None = None,
    threshold: float = 0.5,
) -> dict:
    cfg = yaml.safe_load(open(config))
    d = cfg["data"]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    views = [
        read_clip_frames(p, d["start_frame"], d["end_frame"], d["num_frames"], d["side"])
        for p in clip_paths[: d["max_views"]]
    ]
    while len(views) < d["max_views"]:
        views.append(views[-1].clone())
    batch = torch.stack(views).unsqueeze(0).to(device)  # (1, V, C, T, H, W)

    model = VARSModel(backbone=cfg["video_model"]["backbone"], agg=cfg["video_model"]["agg"])
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.to(device).eval()

    t_off = t_act = 1.0
    if calibration:
        cal = json.load(open(calibration))
        t_off, t_act = cal["temperature_offence"], cal["temperature_action"]

    with torch.no_grad():
        lo, la = model(batch)
    p_off = calibrated_probs(lo, t_off)[0].cpu().numpy()
    p_act = calibrated_probs(la, t_act)[0].cpu().numpy()

    d_off, d_act = decide(p_off, threshold), decide(p_act, threshold)
    overall_conf = min(d_off["confidence"], d_act["confidence"])

    is_dive = ACTION_CLASSES[d_act["pred"]] == "Dive"
    violation = (
        "No violation" if d_off["pred"] == 0 and not is_dive
        else ("Simulation (dive)" if is_dive else "Foul")
    )

    return {
        "violation": violation,
        "action_class": ACTION_CLASSES[d_act["pred"]],
        "card": CARD_NAMES[d_off["pred"]],
        "decision": OFFENCE_SEVERITY_CLASSES[d_off["pred"]],
        "confidence": overall_conf,
        "confidence_breakdown": {
            "offence_severity": d_off["confidence"],
            "action_class": d_act["confidence"],
        },
        "needs_human_review": overall_conf < threshold,
        "explanation": explain_foul(
            d_off["pred"], d_act["pred"], d_off["confidence"], d_act["confidence"]
        ),
        "probabilities": {
            "offence_severity": dict(zip(OFFENCE_SEVERITY_CLASSES, p_off.round(3).tolist())),
            "action_class": dict(zip(ACTION_CLASSES, p_act.round(3).tolist())),
        },
    }


def check_positions(positions_npz: str, config: str = "configs/default.yaml") -> dict:
    """Offside + out-of-bounds from a tracking npz
    (keys: attackers (A,2), defenders (D,2), ball (2,) or ball_seq (T,2))."""
    from .rules.offside import check_offside
    from .rules.out_of_bounds import check_out_of_bounds
    from .rules.pitch import Pitch

    cfg = yaml.safe_load(open(config))["rules"]
    pitch = Pitch(cfg["pitch_length_m"], cfg["pitch_width_m"])
    z = np.load(positions_npz)
    out: dict = {}
    if "attackers" in z and "defenders" in z and "ball" in z:
        r = check_offside(z["attackers"], z["defenders"], z["ball"],
                          pitch, cfg["offside_tolerance_m"])
        out["offside"] = r.__dict__
    if "ball_seq" in z:
        out["out_of_bounds"] = check_out_of_bounds(z["ball_seq"], pitch).__dict__
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--clip", nargs="+", help="one or more camera views of the incident")
    p.add_argument("--checkpoint", default="runs/video/best.pth")
    p.add_argument("--calibration", default=None)
    p.add_argument("--config", default="configs/default.yaml")
    p.add_argument("--positions", default=None, help="tracking npz for offside/OOB")
    p.add_argument("--threshold", type=float, default=0.5)
    args = p.parse_args()

    result: dict = {}
    if args.clip:
        result.update(
            predict_clip(args.clip, args.checkpoint, args.calibration,
                         args.config, threshold=args.threshold)
        )
    if args.positions:
        result["position_checks"] = check_positions(args.positions, args.config)
    print(json.dumps(result, indent=2, default=str))


if __name__ == "__main__":
    main()


Writing src/airef/inference.py


In [ ]:
%%writefile tests/test_smoke.py
"""Smoke tests on synthetic data — no dataset, no GPU needed.

    python -m pytest tests/ -q        (or: python tests/test_smoke.py)
"""
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))


def test_labels_mapping():
    from airef.data.labels import action_class_label, offence_severity_label

    assert offence_severity_label("No offence", "") == 0
    assert offence_severity_label("Offence", "1.0") == 1
    assert offence_severity_label("Offence", "3.0") == 2
    assert offence_severity_label("Offence", "5.0") == 3
    assert offence_severity_label("", "") is None
    assert action_class_label("High leg") == 2
    assert action_class_label("Dont know") is None


def test_offside_geometry():
    from airef.rules.offside import check_offside

    # Clearly offside: attacker 3 m past the second-last defender
    r = check_offside([[23.0, 0.0]], [[50.0, 0.0], [20.0, 0.0]], [10.0, 0.0])
    assert r.offside and r.confidence > 0.5 and not r.needs_human_review
    assert "Offside" in r.explanation

    # Clearly onside: attacker behind the ball and defenders
    r = check_offside([[5.0, 0.0]], [[50.0, 0.0], [20.0, 0.0]], [10.0, 0.0])
    assert not r.offside

    # Marginal: 5 cm — must be flagged for review
    r = check_offside([[20.05, 0.0]], [[50.0, 0.0], [20.0, 0.0]], [10.0, 0.0])
    assert r.needs_human_review

    # Own half — never offside
    r = check_offside([[-5.0, 0.0]], [[50.0, 0.0], [20.0, 0.0]], [-10.0, 0.0])
    assert not r.offside


def test_out_of_bounds():
    from airef.rules.out_of_bounds import check_out_of_bounds

    inside = np.array([[0.0, 0.0], [10.0, 20.0]])
    assert not check_out_of_bounds(inside).out

    over_touch = np.array([[0.0, 33.0], [0.0, 35.0]])
    r = check_out_of_bounds(over_touch)
    assert r.out and r.boundary == "touchline" and r.restart == "throw-in"

    over_goal = np.array([[52.0, 0.0], [53.5, 0.0]])
    r = check_out_of_bounds(over_goal)
    assert r.out and r.boundary == "goal line"


def test_pose_features_shape():
    from airef.pose.features import action_features

    kps = np.random.rand(25, 4, 17, 3).astype(np.float32) * 200
    kps[:, 2:] = np.nan  # only 2 players detected
    f1 = action_features(kps)
    f2 = action_features(np.random.rand(25, 4, 17, 3).astype(np.float32) * 200)
    assert f1.shape == f2.shape and f1.ndim == 1
    assert np.isfinite(f1).all() and np.isfinite(f2).all()


def test_pose_model_train_predict():
    from airef.models.pose_model import PoseRefereeModel

    rng = np.random.default_rng(0)
    X = rng.normal(size=(80, 96)).astype(np.float32)
    y_off = rng.integers(0, 4, 80)
    y_act = rng.integers(0, 8, 80)
    m = PoseRefereeModel("mlp").fit(X, y_off, y_act)
    po, pa = m.predict_proba(X[:5])
    assert po.shape == (5, 4) and pa.shape == (5, 8)
    assert np.allclose(po.sum(1), 1, atol=1e-5)


def test_confidence_and_explanations():
    import torch
    from airef.confidence import decide, fit_temperature
    from airef.explain import explain_foul

    d = decide([0.2, 0.6, 0.1, 0.1])
    assert d["pred"] == 1 and not d["needs_human_review"]
    d = decide([0.3, 0.3, 0.2, 0.2])
    assert d["needs_human_review"]

    logits = torch.randn(50, 4) * 5
    labels = torch.randint(0, 4, (50,))
    t = fit_temperature(logits, labels)
    assert t > 0

    text = explain_foul(3, 2, 0.9, 0.8)
    assert "red card" in text.lower() and "high leg" in text.lower()
    text = explain_foul(1, 0, 0.4, 0.8)
    assert "human review" in text.lower()


def test_video_model_forward():
    """Tiny forward pass with an untrained r3d_18 backbone (CPU-friendly)."""
    import torch
    from airef.models.video_model import VARSModel

    model = VARSModel(backbone="r3d_18", agg="attention", pretrained=False)
    model.eval()
    views = torch.randn(1, 2, 3, 8, 64, 64)  # (B, V, C, T, H, W)
    with torch.no_grad():
        lo, la = model(views)
    assert lo.shape == (1, 4) and la.shape == (1, 8)


if __name__ == "__main__":
    fns = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for fn in fns:
        fn()
        print(f"PASS {fn.__name__}")
    print(f"\n{len(fns)} tests passed.")


Writing tests/test_smoke.py


## 3. Smoke tests (synthetic data — verifies everything before the big download)

In [ ]:
!python tests/test_smoke.py

PASS test_confidence_and_explanations
PASS test_labels_mapping
PASS test_offside_geometry
PASS test_out_of_bounds
PASS test_pose_features_shape
PASS test_pose_model_train_predict
PASS test_video_model_forward

7 tests passed.


## 4. Download SoccerNet-MVFouls (~50 GB)
Enter the NDA password when prompted — **never commit or share it**. Tip: mount Google Drive and set `DATA_ROOT` there so you download once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/soccernet/mvfouls
!cp /content/drive/MyDrive/soccernet_zips/*.zip data/soccernet/mvfouls/

Mounted at /content/drive


In [ ]:
import os, getpass

# Make the airef package findable regardless of earlier cells
if not os.getcwd().endswith('ai-referee'):
    %cd /content/ai-referee
os.environ['PYTHONPATH'] = os.path.abspath('src')

os.environ['SOCCERNET_PASSWORD'] = getpass.getpass('SoccerNet NDA password: ')

DATA_ROOT = 'data/soccernet'   # Colab local disk (no Drive needed)

!python -m airef.data.download --root $DATA_ROOT --task mvfouls --splits train valid test

SoccerNet NDA password: ··········
data/soccernet/mvfouls/train.zip already exists
data/soccernet/mvfouls/valid.zip already exists
data/soccernet/mvfouls/test.zip already exists
Extracting test.zip ...
Extracting train.zip ...
Extracting valid.zip ...


In [ ]:
import zipfile, shutil
from pathlib import Path

mv = Path('data/soccernet/mvfouls')
zips = sorted(mv.glob('*.zip'))
print('found zips:', [z.name for z in zips])

# 1. Remove the wrongly-flattened extraction
for d in mv.glob('action_*'):
    shutil.rmtree(d)
(mv / 'annotations.json').unlink(missing_ok=True)

# 2. Extract each zip into its split folder (train/valid/test)
for zp in zips:
    split = zp.stem.split('_')[0].lower()      # train.zip -> train
    out = mv / split
    out.mkdir(exist_ok=True)
    print(f'extracting {zp.name} -> {out}/')
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(out)
    # If the zip had its own top-level folder (e.g. Train/), flatten it
    if not (out / 'annotations.json').exists():
        subs = [s for s in out.iterdir() if s.is_dir()]
        if len(subs) == 1:
            for item in subs[0].iterdir():
                shutil.move(str(item), out)
            subs[0].rmdir()

!find data/soccernet/mvfouls -maxdepth 2 -name annotations.json

found zips: ['test.zip', 'train.zip', 'valid.zip']
extracting test.zip -> data/soccernet/mvfouls/test/
extracting train.zip -> data/soccernet/mvfouls/train/
extracting valid.zip -> data/soccernet/mvfouls/valid/
data/soccernet/mvfouls/valid/annotations.json
data/soccernet/mvfouls/test/annotations.json
data/soccernet/mvfouls/train/annotations.json


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/soccernet_zips
!cp data/soccernet/mvfouls/*.zip /content/drive/MyDrive/soccernet_zips/
!ls -lh /content/drive/MyDrive/soccernet_zips

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 2.9G
-rw------- 1 root root 256M Jul 16 20:15 test.zip
-rw------- 1 root root 2.3G Jul 16 20:15 train.zip
-rw------- 1 root root 335M Jul 16 20:15 valid.zip


## 5. Explore the annotations

In [ ]:
from collections import Counter
from airef.data.labels import load_split, trainable, OFFENCE_SEVERITY_CLASSES, ACTION_CLASSES

MVFOULS = f'{DATA_ROOT}/mvfouls'
train = trainable(load_split(MVFOULS, 'train'))
print(len(train), 'trainable actions')
print(Counter(OFFENCE_SEVERITY_CLASSES[a.offence_severity] for a in train))
print(Counter(ACTION_CLASSES[a.action_class] for a in train))

2821 trainable actions
Counter({'Offence + No card': 1761, 'Offence + Yellow card': 685, 'No offence': 304, 'Offence + Red card': 71})
Counter({'Standing tackling': 1264, 'Tackling': 447, 'Challenge': 380, 'Holding': 360, 'Elbowing': 178, 'High leg': 103, 'Pushing': 88, 'Dive': 1})


In [ ]:
!du -sh data/soccernet 2>/dev/null || echo "nothing downloaded"
!find data/soccernet -maxdepth 3 | head -20

5.8G	data/soccernet
data/soccernet
data/soccernet/mvfouls
data/soccernet/mvfouls/valid.zip
data/soccernet/mvfouls/valid
data/soccernet/mvfouls/valid/action_407
data/soccernet/mvfouls/valid/action_260
data/soccernet/mvfouls/valid/action_195
data/soccernet/mvfouls/valid/action_321
data/soccernet/mvfouls/valid/action_95
data/soccernet/mvfouls/valid/action_265
data/soccernet/mvfouls/valid/action_162
data/soccernet/mvfouls/valid/action_378
data/soccernet/mvfouls/valid/action_372
data/soccernet/mvfouls/valid/action_364
data/soccernet/mvfouls/valid/action_33
data/soccernet/mvfouls/valid/action_276
data/soccernet/mvfouls/valid/action_235
data/soccernet/mvfouls/valid/action_99
data/soccernet/mvfouls/valid/action_165
data/soccernet/mvfouls/valid/action_132


## 6. Train Pipeline A — VARS-style video model
~2–4 h on L4/A100 for 12 epochs. On a T4, set `batch_size: 2` in `configs/default.yaml`.

In [ ]:
!python -m airef.train_video --config configs/default.yaml --data $DATA_ROOT/mvfouls

/usr/bin/python3: Error while finding module specification for 'airef.train_video' (ModuleNotFoundError: No module named 'airef')


## 7. Train Pipeline B — pose keypoints + classic ML
Keypoint extraction runs once (~1–2 h with GPU) and caches .npz files; training then takes minutes.

In [ ]:
!python -m airef.pose.extract_keypoints --data $DATA_ROOT/mvfouls --out data/keypoints --splits train valid test
!python -m airef.train_pose --config configs/default.yaml --keypoints data/keypoints

## 8. Evaluate & compare both pipelines on the test split

In [ ]:
!python -m airef.evaluate --config configs/default.yaml --data $DATA_ROOT/mvfouls --keypoints data/keypoints --split test
import json; print(json.dumps(json.load(open('runs/comparison_test.json')), indent=2))

## 9. Single-clip inference (the project objective)

In [ ]:
from airef.data.labels import load_split
from airef.inference import predict_clip
import json

action = load_split(MVFOULS, 'test')[0]
result = predict_clip(action.clip_paths, 'runs/video/best.pth',
                      'runs/video/calibration.json', 'configs/default.yaml')
print(json.dumps(result, indent=2))

## 10. Rule-based checks demo — offside & out-of-bounds

In [ ]:
import numpy as np
from airef.rules.offside import check_offside
from airef.rules.out_of_bounds import check_out_of_bounds

r = check_offside(attackers_xy=[[21.0, 5.0]],
                  defenders_xy=[[50.0, 0.0], [20.0, -3.0], [15.0, 8.0]],
                  ball_xy=[10.0, 0.0])
print(r.explanation, '| confidence:', r.confidence)

traj = np.column_stack([np.linspace(0, 5, 25), np.linspace(33.2, 34.6, 25)])
b = check_out_of_bounds(traj)
print(b.explanation, '| confidence:', b.confidence)